# PandaPickCuboid: simulation, PPO, and checkpoint debugging

From the repository root, run `uv sync --locked --extra cuda --group notebooks`
and select that `.venv` as the notebook kernel. On a fresh clone run
`task setup MODE=cuda` first. Restart the kernel after changing GPU visibility.

Run all cells in order. Defaults use four parallel worlds, randomized models,
native MJX-Warp RGB rendering, and a short PPO update, not a learning benchmark.
For CPU debugging set `VISION = False` and `IMPL = "jax"` below (no CUDA extra
required). Increase the explicit run settings for longer experiments.

In [ ]:
import os
import sys
from pathlib import Path

# Works from the repository root or the debugging directory.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "simulation.py").is_file() and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the training repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("JAX_DEFAULT_MATMUL_PRECISION", "highest")
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("WARP_CACHE_PATH", str(ROOT / ".cache/warp"))
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", str(ROOT / ".cache/jax"))
print("Kernel:", sys.executable)

In [ ]:
ENV_NAME = "PandaPickCuboid"
VISION = True
IMPL = "warp"
DOMAIN_RANDOMIZATION = True
SEED = 0
NUM_ENVS = 4
EPISODE_LENGTH = 8
NUM_TIMESTEPS = 16
ROLLOUT_STEPS = 8

In [ ]:
import functools
import json
from datetime import datetime, timezone
import jax
from jax import numpy as jp
import numpy as np
import mediapy as media
import imageio_ffmpeg
from matplotlib import pyplot as plt
from flax import serialization
from brax.training.agents.ppo import train as ppo, networks, networks_vision, checkpoint
from simulation import make_env
from training_cli import parse_args

media.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())
np.set_printoptions(precision=3, suppress=True)
print(jax.devices())

In [ ]:
# Shared validated config and seeded randomization; no Madrona wrapper.
args = parse_args(ENV_NAME, [
    "--smoke", f"--impl={IMPL}", "--vision" if VISION else "--no-vision",
    "--proprioception",
    "--domain_randomization" if DOMAIN_RANDOMIZATION else "--no-domain_randomization",
    f"--seed={SEED}", f"--num_envs={NUM_ENVS}", f"--num_eval_envs={NUM_ENVS}",
    f"--batch_size={NUM_ENVS}", f"--episode_length={EPISODE_LENGTH}",
    f"--num_timesteps={NUM_TIMESTEPS}", f"--rollout_steps={ROLLOUT_STEPS}",
])
env, env_cfg = make_env(args)
eval_env, _ = make_env(args, seed=SEED + 1)
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
print("Action size:", env.action_size, "Episode steps:", env_cfg.episode_length)

In [ ]:
def check_finite(state):
    for leaf in jax.tree.leaves((state.obs, state.reward, state.data.qpos)):
        assert np.isfinite(np.asarray(leaf)).all(), "Nonfinite simulator output"

def pixel_grid(state, max_worlds=4):
    # Only copy the newest RGB frame, not all device state/history.
    images = np.asarray(state.obs["pixels/view_0"][:max_worlds, ..., -3:])
    columns = int(np.ceil(np.sqrt(len(images))))
    rows = int(np.ceil(len(images) / columns))
    h, w = images.shape[1:3]
    grid = np.zeros((rows * h, columns * w, 3), dtype=np.uint8)
    for i, image in enumerate(images):
        r, c = divmod(i, columns)
        grid[r*h:(r+1)*h, c*w:(c+1)*w] = (np.clip(image, 0, 1) * 255).astype(np.uint8)
    return grid

state = jit_reset(jax.random.split(jax.random.PRNGKey(SEED), NUM_ENVS))
check_finite(state)
print("Observations:", jax.tree.map(lambda x: x.shape, state.obs))
if VISION:
    media.show_image(pixel_grid(state), width=512)

In [ ]:
# Inspect randomized parallel worlds under a bounded sinusoidal control signal.
frames, rewards = [], []
for i in range(ROLLOUT_STEPS):
    phase = jp.arange(env.action_size) * 2 * jp.pi / env.action_size
    action = 0.2 * jp.sin(i * env.dt * 2 * jp.pi * 0.2 + phase)
    state = jit_step(state, jp.broadcast_to(action, (NUM_ENVS, env.action_size)))
    check_finite(state)
    rewards.append(np.asarray(state.reward))
    if VISION:
        frames.append(pixel_grid(state))
if frames:
    media.show_video(frames, fps=1 / env.dt, width=512)
plt.plot(np.asarray(rewards))
plt.xlabel("Control step")
plt.ylabel("Reward per world")
plt.show()

## Short PPO run

Training receives already wrapped training and evaluation environments via
`wrap_env=False`. This avoids accidentally wrapping or randomizing twice.
Each run saves in its own directory under `results/notebooks`.

In [ ]:
network_factory = functools.partial(
    networks_vision.make_ppo_networks_vision if VISION else networks.make_ppo_networks,
    policy_hidden_layer_sizes=(32, 32), value_hidden_layer_sizes=(32, 32),
    **(dict(normalise_channels=True, policy_obs_key="_prop", value_obs_key="_prop")
       if VISION else {}),
)
run_dir = ROOT / "results/notebooks" / (
    ENV_NAME + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / "config.json").write_text(json.dumps(env_cfg.to_dict(), indent=2))
progress_rows = []

def progress(step, metrics):
    row = {key: float(value) for key, value in metrics.items()}
    assert all(np.isfinite(v) for v in row.values()), row
    progress_rows.append({"step": int(step), **row})
    print("Step", step, row)

make_inference_fn, params, metrics = ppo.train(
    environment=env, eval_env=eval_env, wrap_env=False,
    max_devices_per_host=1, use_pmap_on_reset=True,
    vision=VISION, augment_pixels=VISION, network_factory=network_factory,
    num_timesteps=args.num_timesteps, episode_length=env_cfg.episode_length,
    action_repeat=args.action_repeat, num_envs=NUM_ENVS, num_eval_envs=NUM_ENVS,
    num_evals=1, num_resets_per_eval=1,
    unroll_length=args.unroll_length, batch_size=args.batch_size,
    num_minibatches=1, num_updates_per_batch=1,
    learning_rate=args.learning_rate, entropy_cost=args.entropy_cost,
    discounting=args.discounting, reward_scaling=args.reward_scaling,
    max_grad_norm=args.max_grad_norm, clipping_epsilon=args.clipping_epsilon,
    normalize_observations=args.normalize_observations,
    deterministic_eval=True, seed=SEED, progress_fn=progress,
    save_checkpoint_path=str(run_dir / "checkpoints"),
)
(run_dir / "metrics.json").write_text(json.dumps(progress_rows, indent=2))
print("Saved run:", run_dir)

In [ ]:
# Reload this run's checkpoint, not incompatible historical policy files.
latest = max((p for p in (run_dir / "checkpoints").iterdir()
              if p.is_dir() and p.name.isdigit()), key=lambda p: int(p.name))
loaded_params = checkpoint.load(str(latest))
original_leaves = jax.tree.leaves(params)
restored_leaves = jax.tree.leaves(loaded_params)
assert len(original_leaves) == len(restored_leaves)
for before, after in zip(original_leaves, restored_leaves):
    np.testing.assert_array_equal(np.asarray(before), np.asarray(after))

# Optional portable Flax serialization uses the SAME filename and template.
params_path = run_dir / "params.msgpack"
params_path.write_bytes(serialization.to_bytes(params))
msgpack_params = serialization.from_bytes(params, params_path.read_bytes())
for before, after in zip(original_leaves, jax.tree.leaves(msgpack_params)):
    np.testing.assert_array_equal(np.asarray(before), np.asarray(after))
# Build inference from the reloaded checkpoint, not the original params.
jit_policy = jax.jit(make_inference_fn(loaded_params, deterministic=True))
print("Verified checkpoint and msgpack round trips:", latest)

In [ ]:
state = jit_reset(jax.random.split(jax.random.PRNGKey(SEED + 2), NUM_ENVS))
rng = jax.random.PRNGKey(SEED + 3)
frames, rewards = [], []
for _ in range(ROLLOUT_STEPS):
    rng, action_key = jax.random.split(rng)
    # Brax's batched policy expects one PRNG key, not a key per world.
    action, _ = jit_policy(state.obs, action_key)
    assert np.isfinite(np.asarray(action)).all()
    state = jit_step(state, action)
    check_finite(state)
    rewards.append(np.asarray(state.reward))
    if VISION:
        frames.append(pixel_grid(state))
if frames:
    media.write_video(run_dir / "restored-policy.mp4", frames, fps=1 / env.dt)
    media.show_video(frames, fps=1 / env.dt, width=512)
np.savez(run_dir / "rollout.npz", rewards=np.asarray(rewards), qpos=np.asarray(state.data.qpos))
plt.plot(np.asarray(rewards))
plt.xlabel("Control step")
plt.ylabel("Restored-policy reward per world")
plt.show()

In [ ]:
# Inspect policy parameter shapes without dumping large tensors.
from flax.traverse_util import flatten_dict
for key, value in flatten_dict(loaded_params[1]).items():
    print("/".join(key), np.shape(value))
print("Completed simulation, PPO, checkpoint reload, and policy rollout:", run_dir)